This section includes loading and reading teh data to be used

In [10]:
import urllib.request
import torch
import torch.nn as nn

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)

file_path = "the-verdict.txt"

urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7b22111096e0>)

This code block counts the nunber of characters in the text to be used

In [11]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Number of characters:", len(raw_text))
print(raw_text[:100])

Number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


This section uses regular expression to split texts based on punctuations and white spaces to create tokens,

Note: this is not the usual mathod of creating tokens, but for easier understanding, this is used to create an idea of what a token is, a token is averagelly 3/4 of a word, not a word.

In [12]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

preprocessed = [
    item.strip()
    for item in preprocessed
    if item.strip()
]

print("Number of tokens:", len(preprocessed))
print(preprocessed[:30])

Number of tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


The cde block below shows the vocabulary used, each character, word, punctuatrion and element represent the token we have.

In [13]:
all_words = sorted(set(preprocessed))

vocab = {
    token: integer
    for integer, token in enumerate(all_words)
}

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size: 1130
[('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9), ('?', 10), ('A', 11), ('Ah', 12), ('Among', 13), ('And', 14), ('Are', 15), ('Arrt', 16), ('As', 17), ('At', 18), ('Be', 19)]


The code below extracts the token IDs for each token.

In [14]:
token_ids = [vocab[token] for token in preprocessed]

print(token_ids[:30])

[53, 44, 149, 1003, 57, 38, 818, 115, 256, 486, 6, 1002, 115, 500, 435, 392, 6, 908, 585, 1077, 709, 508, 961, 1016, 663, 1016, 535, 987, 5, 568]


The token created earlier is converted to a pytorch tensor that will be used when implementing the code.

In [15]:
token_ids = torch.tensor(token_ids)

print(token_ids.shape)
print(token_ids[:30])

torch.Size([4690])
tensor([  53,   44,  149, 1003,   57,   38,  818,  115,  256,  486,    6, 1002,
         115,  500,  435,  392,    6,  908,  585, 1077,  709,  508,  961, 1016,
         663, 1016,  535,  987,    5,  568])


The code below uses Pytorch nn.Embedding method to create an embedding for the vocabullary collected.

In [16]:
vocab_size = len(vocab)
embedding_dim = 256

token_embedding = nn.Embedding(
    vocab_size,
    embedding_dim
)

tok_emb = token_embedding(token_ids[:4])

print("Token embedding shape:", tok_emb.shape)

Token embedding shape: torch.Size([4, 256])


The positional embedding also uses nn.Embedding to create the positional embedding.

In [17]:
context_length = 4

position_embedding = nn.Embedding(
    context_length,
    embedding_dim
)

position_ids = torch.arange(context_length)

pos_emb = position_embedding(position_ids)

print("Position IDs:", position_ids)
print("Position embedding shape:", pos_emb.shape)

Position IDs: tensor([0, 1, 2, 3])
Position embedding shape: torch.Size([4, 256])


The code below combines the token and positional embedding to craete an embedding information that can then be passed into the transformer  architecture.

In [18]:
x = tok_emb + pos_emb

print("Final embedding shape:", x.shape)

Final embedding shape: torch.Size([4, 256])
